In [120]:
import pandas as pd
import os
import glob
import re
os.getcwd()
#os.listdir()

'/Users/avinash.anand/Documents/data-engg/Ecom-project'

In [121]:
file_dir = f'{os.getcwd()}/datafiles'
print(file_dir)

/Users/avinash.anand/Documents/data-engg/Ecom-project/datafiles


In [122]:
# get all the csv filenames in the folder
csvfiles = [re.split('/', filepath)[-1] for filepath in glob.glob(f'{file_dir}/**.csv', recursive=True)]
print(csvfiles)

['olist_sellers_dataset.csv', 'product_category_name_translation.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv']


In [123]:
# Alternate way to get the csv file names from the folder
# Using os module

csv_files = [f for f in os.listdir(file_dir) if f.endswith('.csv')]
print(csv_files)

['olist_sellers_dataset.csv', 'product_category_name_translation.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv']


In [124]:
#create a dictionary of csvfiles
f_dict = {f.replace("olist_","").replace(".csv","").replace("_dataset","").replace("_translation",""): f for f in csvfiles}
for name, value in f_dict.items():
    print(f"{name}: {value}")

sellers: olist_sellers_dataset.csv
product_category_name: product_category_name_translation.csv
orders: olist_orders_dataset.csv
order_items: olist_order_items_dataset.csv
customers: olist_customers_dataset.csv
geolocation: olist_geolocation_dataset.csv
order_payments: olist_order_payments_dataset.csv
order_reviews: olist_order_reviews_dataset.csv
products: olist_products_dataset.csv


In [125]:
# Get filepath which contains filename with its directory

def get_filepath(filename): return f'{file_dir}/{filename}'

get_filepath(f_dict['sellers'])

'/Users/avinash.anand/Documents/data-engg/Ecom-project/datafiles/olist_sellers_dataset.csv'

In [126]:
# Get shape of each file

#df = pd.read_csv(f'{file_dir}/{f_dict['sellers']}')
#print(df.shape)

def get_shape(df, filename):
    print(f'Shape of {filename} is {df.shape}')

# Print the shape of the files
for filename in f_dict.values():
    df = pd.read_csv(get_filepath(filename))
    get_shape(df, filename)

Shape of olist_sellers_dataset.csv is (3095, 4)
Shape of product_category_name_translation.csv is (71, 2)
Shape of olist_orders_dataset.csv is (99441, 8)
Shape of olist_order_items_dataset.csv is (112650, 7)
Shape of olist_customers_dataset.csv is (99441, 5)
Shape of olist_geolocation_dataset.csv is (1000163, 5)
Shape of olist_order_payments_dataset.csv is (103886, 5)
Shape of olist_order_reviews_dataset.csv is (99224, 7)
Shape of olist_products_dataset.csv is (32951, 9)


In [127]:
# Get column names from the files

columns = pd.read_csv(get_filepath(f_dict['sellers']), nrows=0).columns.tolist()
print(columns)

['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


In [128]:
# Create JSON data files from the CSV files

json_dir = f'{os.getcwd()}/datafile-json'

for name, filename in f_dict.items():
    df = pd.read_csv(get_filepath(filename=filename))

    #convert into json
    df.to_json(f'{json_dir}/{name}.json', orient='records', lines=True)

In [129]:
# create a schema.json file which contains the csv file metadata
'''
file = f_dict['sellers']
df = pd.read_csv(get_filepath(file))
columns = df.columns.tolist()
df.dtypes
'''

def pandas_dtype_to_sql(dtype):
    dtype = str(dtype)
    if dtype.startswith('int'):
        return 'INTEGER'
    elif dtype.startswith('float'):
        return 'FLOAT'
    elif dtype.startswith('bool'):
        return 'BOOLEAN'
    elif dtype.startswith('datetime'):
        return 'TIMESTAMP'
    else:
        return 'VARCHAR'  # object/string fallback

for col, dtype in df.dtypes.items():
    pass #print(col, pandas_dtype_to_sql(dtype))

schemas = {}
for name, filename in f_dict.items():
    df = pd.read_csv(get_filepath(filename))
    schemas[name] = {
        "row_count": df.shape[0],
        "columns": [
            {"column_name": col, "datatype":pandas_dtype_to_sql(dtype), "column_position": i}
            for i, (col, dtype) in enumerate(df.dtypes.items(), start=1)
        ]
    }

schemas

{'sellers': {'row_count': 3095,
  'columns': [{'column_name': 'seller_id',
    'datatype': 'VARCHAR',
    'column_position': 1},
   {'column_name': 'seller_zip_code_prefix',
    'datatype': 'INTEGER',
    'column_position': 2},
   {'column_name': 'seller_city', 'datatype': 'VARCHAR', 'column_position': 3},
   {'column_name': 'seller_state',
    'datatype': 'VARCHAR',
    'column_position': 4}]},
 'product_category_name': {'row_count': 71,
  'columns': [{'column_name': 'product_category_name',
    'datatype': 'VARCHAR',
    'column_position': 1},
   {'column_name': 'product_category_name_english',
    'datatype': 'VARCHAR',
    'column_position': 2}]},
 'orders': {'row_count': 99441,
  'columns': [{'column_name': 'order_id',
    'datatype': 'VARCHAR',
    'column_position': 1},
   {'column_name': 'customer_id', 'datatype': 'VARCHAR', 'column_position': 2},
   {'column_name': 'order_status',
    'datatype': 'VARCHAR',
    'column_position': 3},
   {'column_name': 'order_purchase_timestam

In [130]:
import json

with open(f'{os.getcwd()}/schemas.json', mode='w') as f_json:
    json.dump(schemas, f_json, indent=2)

with open(f'{os.getcwd()}/schemas.json', mode='r') as f_json:
    schema = json.load(f_json)
    print(schema)

{'sellers': {'row_count': 3095, 'columns': [{'column_name': 'seller_id', 'datatype': 'VARCHAR', 'column_position': 1}, {'column_name': 'seller_zip_code_prefix', 'datatype': 'INTEGER', 'column_position': 2}, {'column_name': 'seller_city', 'datatype': 'VARCHAR', 'column_position': 3}, {'column_name': 'seller_state', 'datatype': 'VARCHAR', 'column_position': 4}]}, 'product_category_name': {'row_count': 71, 'columns': [{'column_name': 'product_category_name', 'datatype': 'VARCHAR', 'column_position': 1}, {'column_name': 'product_category_name_english', 'datatype': 'VARCHAR', 'column_position': 2}]}, 'orders': {'row_count': 99441, 'columns': [{'column_name': 'order_id', 'datatype': 'VARCHAR', 'column_position': 1}, {'column_name': 'customer_id', 'datatype': 'VARCHAR', 'column_position': 2}, {'column_name': 'order_status', 'datatype': 'VARCHAR', 'column_position': 3}, {'column_name': 'order_purchase_timestamp', 'datatype': 'VARCHAR', 'column_position': 4}, {'column_name': 'order_approved_at'